In [ ]:
import functions_to_use as fu
import pandas as pd
import re
import datetime as dt
from sklearn.preprocessing import LabelEncoder
from pytorch_tabnet.tab_model import TabNetClassifier


In [ ]:
cols = "C:\\Users\\YOGA\\Desktop\\master s4 PFE\\project\\IDS-MAS\\new-test\\model\\cols_file.txt"

data = pd.read_csv("C:\\Users\\YOGA\\Desktop\\master s4 PFE\\project\\IDS-MAS\\data\\friday.csv")

rows = data.head(1)


,id,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Total Fwd Packet,...,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,ICMP Code,ICMP Type,Total TCP Flow Time,Label,Attempted Category
62556,62557,192.168.10.5-205.174.165.73-53709-8080-6,192.168.10.5,53709,205.174.165.73,8080,6,2017-07-07 13:39:36.267908,81379,5,...,0,0.0,0.0,0,0,-1,-1,81379,Botnet,-1
63141,63142,192.168.10.5-205.174.165.73-53710-8080-6,192.168.10.5,53710,205.174.165.73,8080,6,2017-07-07 13:39:46.349992,65872,5,...,0,0.0,0.0,0,0,-1,-1,65872,Botnet,-1
65822,65823,192.168.10.5-205.174.165.73-53885-8080-6,192.168.10.5,53885,205.174.165.73,8080,6,2017-07-07 13:49:32.212213,65336,5,...,0,0.0,0.0,0,0,-1,-1,65336,Botnet,-1
66043,66044,192.168.10.5-205.174.165.73-53888-8080-6,192.168.10.5,53888,205.174.165.73,8080,6,2017-07-07 13:49:42.274615,210485,6,...,0,0.0,0.0,0,0,-1,-1,210485,Botnet,-1
66454,66455,192.168.10.5-205.174.165.73-53892-8080-6,192.168.10.5,53892,205.174.165.73,8080,6,2017-07-07 13:50:12.758925,60046,5,...,0,0.0,0.0,0,0,-1,-1,60046,Botnet,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70190,70191,172.16.0.1-192.168.10.50-59328-7741-6,172.16.0.1,59328,192.168.10.50,7741,6,2017-07-07 17:52:25.300815,49,1,...,0,0.0,0.0,0,0,-1,-1,49,Portscan,-1
70191,70192,172.16.0.1-192.168.10.50-46948-1071-6,172.16.0.1,46948,192.168.10.50,1071,6,2017-07-07 17:54:24.406921,72,1,...,0,0.0,0.0,0,0,-1,-1,72,Portscan,-1
70192,70193,172.16.0.1-192.168.10.50-39700-4003-6,172.16.0.1,39700,192.168.10.50,4003,6,2017-07-07 17:55:02.475117,49,1,...,0,0.0,0.0,0,0,-1,-1,49,Portscan,-1
70193,70194,172.16.0.1-192.168.10.50-38096-691-6,172.16.0.1,38096,192.168.10.50,691,6,2017-07-07 17:54:44.105011,42,1,...,0,0.0,0.0,0,0,-1,-1,42,Portscan,-1


In [ ]:

rows_data = []
columns = fu.extract_columns(cols)

data = data[data['Label']!='BENIGN']

data = data[data['Label']!='Botnet - Attempted']

data.head(200)


In [3]:
import pickle as pkl
with open("C:\\Users\\YOGA\\Desktop\\master s4 PFE\\project\\IDS-MAS\\new-test\\model\\encoder_x.pkl", "rb") as f:
    label_encoder = pkl.load(f)
    
# with open("C:\\Users\\YOGA\\Desktop\\master s4 PFE\\project\\IDS-MAS\\new-test\\model\\encoder_y.pkl", "rb") as f:
#     label_encoder = pkl.load(f)

label_encoder.classes_


array(['1.1.70.73', '1.193.219.21', '1.193.219.24', ..., '99.192.248.32',
       '99.193.229.25', '99.224.25.39'], shape=(19044,), dtype=object)

In [4]:
encoded_row = fu.encoder(data=data, encoder=label_encoder)

encoded_row

{'id': 62557.0,
 'Flow ID': 203713.0,
 'Src IP': 3.0,
 'Src Port': 53709.0,
 'Dst IP': 1.0,
 'Dst Port': 8080.0,
 'Protocol': 6.0,
 'Timestamp': 391.0,
 'Flow Duration': 81379.0,
 'Total Fwd Packet': 5.0,
 'Total Bwd packets': 4.0,
 'Total Length of Fwd Packet': 196.0,
 'Total Length of Bwd Packet': 128.0,
 'Fwd Packet Length Max': 196.0,
 'Fwd Packet Length Min': 0.0,
 'Fwd Packet Length Mean': 39.2,
 'Fwd Packet Length Std': 87.65386471799175,
 'Bwd Packet Length Max': 128.0,
 'Bwd Packet Length Min': 0.0,
 'Bwd Packet Length Mean': 32.0,
 'Bwd Packet Length Std': 64.0,
 'Flow Bytes/s': 3981.371115398321,
 'Flow Packets/s': 110.5936420943978,
 'Flow IAT Mean': 10172.375,
 'Flow IAT Std': 27662.385301227576,
 'Flow IAT Max': 78629.0,
 'Flow IAT Min': 31.0,
 'Fwd IAT Total': 81379.0,
 'Fwd IAT Mean': 20344.75,
 'Fwd IAT Std': 39749.43681257216,
 'Fwd IAT Max': 79967.0,
 'Fwd IAT Min': 31.0,
 'Bwd IAT Total': 80830.0,
 'Bwd IAT Mean': 26943.333333333336,
 'Bwd IAT Std': 44763.5520969162

In [1]:
# define new model with basic parameters and load state dict weights
import pickle as pkl
# The name of the file to save to
model_file = './model/model.pkl'

# Open the file in binary read mode and load the variable
with open(model_file, 'rb') as file:
    loaded_clf = pkl.load(file)
loaded_clf

,n_d,8
,n_a,8
,n_steps,3
,gamma,1.3
,cat_idxs,[]
,cat_dims,[]
,cat_emb_dim,[]
,n_independent,2
,n_shared,2
,epsilon,1e-15
,momentum,0.02


In [6]:
from sklearn.utils.validation import check_is_fitted
check_is_fitted(loaded_clf)

In [7]:
import numpy as np
encoded_row

{'id': 62557.0,
 'Flow ID': 203713.0,
 'Src IP': 3.0,
 'Src Port': 53709.0,
 'Dst IP': 1.0,
 'Dst Port': 8080.0,
 'Protocol': 6.0,
 'Timestamp': 391.0,
 'Flow Duration': 81379.0,
 'Total Fwd Packet': 5.0,
 'Total Bwd packets': 4.0,
 'Total Length of Fwd Packet': 196.0,
 'Total Length of Bwd Packet': 128.0,
 'Fwd Packet Length Max': 196.0,
 'Fwd Packet Length Min': 0.0,
 'Fwd Packet Length Mean': 39.2,
 'Fwd Packet Length Std': 87.65386471799175,
 'Bwd Packet Length Max': 128.0,
 'Bwd Packet Length Min': 0.0,
 'Bwd Packet Length Mean': 32.0,
 'Bwd Packet Length Std': 64.0,
 'Flow Bytes/s': 3981.371115398321,
 'Flow Packets/s': 110.5936420943978,
 'Flow IAT Mean': 10172.375,
 'Flow IAT Std': 27662.385301227576,
 'Flow IAT Max': 78629.0,
 'Flow IAT Min': 31.0,
 'Fwd IAT Total': 81379.0,
 'Fwd IAT Mean': 20344.75,
 'Fwd IAT Std': 39749.43681257216,
 'Fwd IAT Max': 79967.0,
 'Fwd IAT Min': 31.0,
 'Bwd IAT Total': 80830.0,
 'Bwd IAT Mean': 26943.333333333336,
 'Bwd IAT Std': 44763.5520969162

In [8]:
f_to_keep = fu.extract_columns("./model/cols_file.txt")
len(f_to_keep)

59

In [9]:
# "Keep k if k is NOT in the removal list"
encoded_row = {f: v for f, v in encoded_row.items() if f in f_to_keep}
type(encoded_row)


dict

In [10]:
encoded_pd = pd.DataFrame([encoded_row])
encoded_pd = encoded_pd.astype(np.float32)

In [11]:
encoded_pd

,Src Port,Dst Port,Protocol,Flow Duration,Total Fwd Packet,Total Bwd packets,Total Length of Fwd Packet,Total Length of Bwd Packet,Fwd Packet Length Max,Fwd Packet Length Min,...,Bwd Init Win Bytes,Fwd Act Data Pkts,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,ICMP Code,ICMP Type,Total TCP Flow Time
0,53709.0,8080.0,6.0,81379.0,5.0,4.0,196.0,128.0,196.0,0.0,...,237.0,1.0,20.0,0.0,0.0,0.0,0.0,-1.0,-1.0,81379.0


In [12]:
print(check_is_fitted(loaded_clf))

None


In [13]:
predictions = loaded_clf.predict(encoded_pd.values)
predictions

array([1])

In [15]:
# Read y_encoder to read Label as text
import pickle as pkl
file_path = "C:\\Users\\YOGA\\Desktop\\master s4 PFE\\project\\IDS-MAS\\new-test\\model\\encoder_y.pkl"
with open(file_path,'rb') as file:
    y_encoder = pkl.load(file)

y_encoder.inverse_transform([0])
    

array(['BENIGN'], dtype=object)

In [16]:
status = y_encoder.inverse_transform([0])
status[0]

'BENIGN'

In [17]:
y_encoder.classes_

array(['BENIGN', 'Botnet', 'DDoS', 'DoS', 'FTP-Patator', 'Heartbleed',
       'Infiltration', 'Port Scanning', 'SSH-Patator', 'Web Attacks'],
      dtype=object)

In [20]:
encoded_row['status']

KeyError: 'status'